In [ ]:
# %% [markdown]
# # Automatic Dysarthria Detection and Severity Classification
# 
# This Colab-ready script implements an end-to-end pipeline for:
# 1. Data preparation for TORGO and UA Speech
# 2. MFCC + SVM baseline
# 3. wav2vec2 fine-tuning (binary + severity)
# 4. Cross-dataset evaluation
# 5. Basic interpretability
# 6. Final structured report generation
# 
# > Notes:
# > - Designed for practical execution on limited GPU resources.
# > - Uses max-duration clipping and layer freezing for efficiency.
# > - Expected audio sample rate is standardized to 16kHz.

# %% [markdown]
# ## 0) Install dependencies (Colab)

# %%
# If running in Google Colab, use pinned versions (avoid pandas>=3 conflicts).
# Set RUN_INSTALL = True only when you need to install/repair environment.
RUN_INSTALL = False
if RUN_INSTALL:
    import subprocess
    import sys

    # Pin transformers to a Colab-stable combo. Newer Colab images sometimes ship a
    # half-upgraded transformers tree that breaks with: cannot import is_tf_tensor.
    pkgs = [
        "pandas==2.2.2",
        "scikit-learn==1.6.1",
        "imbalanced-learn==0.13.0",
        "librosa==0.10.2.post1",
        "soundfile==0.13.1",
        # 4.35.x avoids Colab mixed-wheel issues with is_tf_tensor / utils.
        "transformers==4.35.2",
        "tokenizers==0.15.2",
        "huggingface_hub==0.19.4",
        "datasets==2.21.0",
        "accelerate==0.24.1",
        "matplotlib==3.10.0",
        "seaborn==0.13.2",
        "tqdm==4.67.1",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# %% [markdown]
# ## 0b) Repair broken `transformers` (Colab) — run if you see `is_tf_tensor` ImportError
#
# Symptom: `ImportError: cannot import name 'is_tf_tensor' from 'transformers.utils'`
#
# Fix: set `REPAIR_TRANSFORMERS = True` below, run this cell, allow auto-restart (or **Runtime → Restart session**), then set it back to **False** and run from section 1.

# %%
# Colab: set True once per fresh runtime if imports fail with `is_tf_tensor`; then False.
REPAIR_TRANSFORMERS = False
if REPAIR_TRANSFORMERS:
    import subprocess
    import sys

    # Colab ships a partial upgrade tree; uninstall a few times (pip may leave remnants).
    for pkg in ("transformers", "tokenizers", "accelerate", "huggingface-hub"):
        for _ in range(4):
            subprocess.call(
                [sys.executable, "-m", "pip", "uninstall", "-y", pkg],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            "transformers==4.35.2",
            "tokenizers==0.15.2",
            "huggingface_hub==0.19.4",
            "accelerate==0.24.1",
            "safetensors>=0.4.3",
        ]
    )

    # Verify in a subprocess (fresh interpreter, no stale sys.modules in this kernel).
    subprocess.check_call(
        [
            sys.executable,
            "-c",
            "import transformers; "
            "import transformers.feature_extraction_sequence_utils; "
            "from transformers import Wav2Vec2ForSequenceClassification; "
            "print('transformers OK:', transformers.__version__)",
        ]
    )

    print(
        "transformers repair finished.\n"
        "If the next cell still errors, use Runtime → Restart session once, "
        "then set REPAIR_TRANSFORMERS = False and run from section 1."
    )

    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None and ip.kernel is not None:
            print("Auto-restarting runtime so imports pick up the new wheels…")
            ip.kernel.do_shutdown(restart=True)
    except Exception:
        pass

# %% [markdown]
# ## 1) Imports and global config

# %%
import os
import re
import json
import math
import wave
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from tqdm.auto import tqdm

import librosa
import soundfile as sf

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC

from imblearn.over_sampling import RandomOverSampler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# Fail fast on Colab's broken mixed transformers installs (error appears at wav2vec load otherwise).
try:
    import transformers.feature_extraction_sequence_utils as _transformers_fe_sanity  # noqa: F401
except ImportError as _e:
    if "is_tf_tensor" in str(_e) or "transformers.utils" in str(_e):
        raise ImportError(
            "Broken `transformers` install (common on Colab).\n"
            "Fix: set REPAIR_TRANSFORMERS = True in section 0b, run that cell, then "
            "Runtime → Restart session, set it back to False, and re-run from section 1.\n"
            f"Original: {_e}"
        ) from _e
    raise

warnings.filterwarnings("ignore")

SEED = 42
TARGET_SR = 16000
MAX_AUDIO_SEC = 5.0
MAX_AUDIO_LEN = int(TARGET_SR * MAX_AUDIO_SEC)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    # Faster matmul on Ampere+ (safe for training); no-op on older GPUs.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"Using device: {DEVICE}")

# %% [markdown]
# ## 2) Configuration
# 
# Set these paths according to your Colab storage layout.

# %%
CONFIG = {
    # Point these at folders that **contain .wav files** (extracted datasets), or leave defaults
    # and use auto-discovery from Google Drive / /content (see setup cell below).
    "torgo_root": "/content/datasets/torgo-audio",
    "ua_root": "/content/datasets/noise-reduced-uaspeech-dysarthria-dataset",
    # Colab: mount Drive and search under these roots for folder names matching TORGO / UA Speech.
    "auto_mount_google_drive": False,
    "auto_discover_dataset_paths": False,
    # Narrow this to e.g. ["/content/drive/MyDrive/MyDatasets"] to speed up search on large Drives.
    "dataset_search_roots": ["/content/drive/MyDrive", "/content"],
    "dataset_search_max_depth": 6,
    # Optional metadata CSV with columns:
    # [audio_path, dataset, binary_label, severity_label, speaker_id]
    "metadata_csv": None,  # e.g., "/content/data/metadata.csv"
    "output_dir": "/content/dysarthria_outputs",
    "wav2vec_checkpoint": "facebook/wav2vec2-base",
    # Required for attention rollout: SDPA / flash_attn return None for output_attentions.
    "wav2vec_attn_implementation": "eager",
    # VRAM: wav2vec2-base + 5s@16k is light; ~1.5GB @ bs=4 → use 16–24 on a 15GB GPU. If OOM, lower batch_size.
    "batch_size":8,
    # Colab: audio decoding in __getitem__ is CPU-heavy; multiprocessing workers often make epochs *slower*.
    # Use 0 (default) or 2. Only raise if you verified a speedup on your machine.
    "num_workers": 0,
    # Mixed precision: usually faster on T4/L4/A100 and lower VRAM.
    "use_amp": True,
    # Effective batch = batch_size * gradient_accumulation_steps (same memory as batch_size alone).
    "gradient_accumulation_steps": 1,
    # Reduce defaults for faster iteration; increase for final runs.
    "epochs_binary": 3,
    "epochs_severity": 3,
    # Base LR at lr_ref_batch_size; scaled when auto_scale_lr_with_batch is True.
    "lr_base": 2e-5,
    "lr_ref_batch_size": 4,
    "auto_scale_lr_with_batch": True,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "freeze_feature_encoder": True,
    "freeze_encoder_layers": 6,  # freeze first N transformer layers
    # Keep runtime practical on Colab. None means use all discovered files.
    "max_samples_per_dataset": 12000,
    # UA Speech severity mapping (dysarthric speakers only).
    # Labels use 0..3 for [very_low, low, medium, high].
    # IMPORTANT: adjust if your UA package includes official severity metadata.
    "ua_speaker_severity_map": {
        "M01": 0, "M04": 0, "M05": 0, "F02": 0,
        "M07": 1, "M08": 1, "M09": 1, "F03": 1,
        "M10": 2, "M11": 2, "M12": 2, "F04": 2,
        "M14": 3, "M16": 3, "F05": 3,
    },
    # Severity robustness controls for noisy/partial label packaging.
    "severity_expected_num_classes": 4,
    "severity_split_max_tries": 40,
    "severity_allow_collapse_4_to_3": True,
    # Interpretability can be expensive on large test sets.
    "enable_interpretability": False,
    "perm_n_repeats": 1,
    "perm_max_samples": 1500,
    "perm_n_jobs": -1,
    # Attention rollout maps: one forward pass per sample (cheap vs permutation importance).
    "enable_attention_maps": True,
    "attention_num_samples": 1,
    "attention_save_plots": True,
}

if CONFIG.get("auto_scale_lr_with_batch", True):
    CONFIG["lr"] = float(CONFIG["lr_base"]) * (float(CONFIG["batch_size"]) / float(CONFIG["lr_ref_batch_size"]))
else:
    CONFIG["lr"] = float(CONFIG.get("lr", CONFIG["lr_base"]))

Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
print(
    f"Output folder (models, report, CSV): {CONFIG['output_dir']}\n"
    "  It stays empty until training/report cells finish successfully."
)
print(
    f"wav2vec2: batch_size={CONFIG['batch_size']}, "
    f"grad_accum={CONFIG.get('gradient_accumulation_steps', 1)}, "
    f"lr={CONFIG['lr']:.2e}, num_workers={CONFIG['num_workers']}, "
    f"use_amp={CONFIG.get('use_amp', True)}"
)


def print_env_versions() -> None:
    print("\nEnvironment versions")
    print("--------------------")
    print(f"python: {os.sys.version.split()[0]}")
    print(f"pandas: {pd.__version__}")
    print(f"torch: {torch.__version__}")


print_env_versions()

# %% [markdown]
# ## 3) Utility functions

# %%
SEVERITY_KEYWORDS = {
    "very_low": 0,
    "low": 1,
    "medium": 2,
    "high": 3,
}


def normalize_audio(x: np.ndarray) -> np.ndarray:
    peak = np.max(np.abs(x)) + 1e-9
    return x / peak


def _load_with_wave_stdlib(path: str) -> Tuple[np.ndarray, int]:
    """PCM RIFF WAV via stdlib (sometimes works when libsndfile rejects headers)."""
    with wave.open(path, "rb") as wf:
        sr = wf.getframerate()
        n_ch = wf.getnchannels()
        sw = wf.getsampwidth()
        n_frames = wf.getnframes()
        raw = wf.readframes(n_frames)
    if sw == 1:
        audio = np.frombuffer(raw, dtype=np.uint8).astype(np.float32)
        audio = (audio - 128.0) / 128.0
    elif sw == 2:
        audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    elif sw == 4:
        audio = np.frombuffer(raw, dtype=np.int32).astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f"Unsupported WAV sample width: {sw}")
    if n_ch > 1:
        audio = audio.reshape(-1, n_ch).mean(axis=1)
    return audio.astype(np.float32), sr


def _load_with_torchaudio(path: str) -> Tuple[np.ndarray, int]:
    import torchaudio

    wav, sr = torchaudio.load(path)
    audio = wav.mean(dim=0).detach().cpu().numpy().astype(np.float32)
    return audio, int(sr)


def load_audio_mono_16k(path: str, target_sr: int = TARGET_SR) -> np.ndarray:
    """
    Load mono audio resampled to target_sr, fixed length. Tries several backends
    because some Kaggle TORGO files trip libsndfile ('Format not recognised').
    """
    path = str(path)
    sz = Path(path).stat().st_size
    if sz < 128:
        raise ValueError(f"Audio file too small ({sz} bytes): {path}")

    audio: Optional[np.ndarray] = None
    sr: Optional[int] = None
    errors: List[str] = []

    try:
        audio, sr = sf.read(path)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        audio = audio.astype(np.float32)
    except Exception as e:
        errors.append(f"soundfile: {e}")

    if audio is None:
        try:
            audio, sr = _load_with_wave_stdlib(path)
        except Exception as e:
            errors.append(f"wave: {e}")

    if audio is None:
        try:
            audio, sr = librosa.load(path, sr=None, mono=True)
            audio = audio.astype(np.float32)
            sr = int(sr)
        except Exception as e:
            errors.append(f"librosa: {e}")

    if audio is None:
        try:
            audio, sr = _load_with_torchaudio(path)
        except Exception as e:
            errors.append(f"torchaudio: {e}")

    if audio is None or sr is None:
        raise RuntimeError(f"Could not decode audio {path}. Tried: {'; '.join(errors)}")

    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
    audio = normalize_audio(audio.astype(np.float32))
    if len(audio) >= MAX_AUDIO_LEN:
        audio = audio[:MAX_AUDIO_LEN]
    else:
        pad = MAX_AUDIO_LEN - len(audio)
        audio = np.pad(audio, (0, pad), mode="constant")
    return audio


def metric_bundle(y_true: List[int], y_pred: List[int], average: str = "binary") -> Dict[str, float]:
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


def print_metrics_table(name: str, metrics: Dict[str, float]) -> None:
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")


def infer_speaker_id_from_path(path: str) -> str:
    p = Path(path)
    parts = [x for x in p.parts if x]
    # UA examples: .../noisereduced-uaspeech/M01/*.wav or .../control/CM01/*.wav
    for token in reversed(parts):
        # UA speaker folders: M01, F02, CM01, CF02
        if re.fullmatch(r"(?:C?[FM]\d{2}|[FM]C\d{2})", token):
            return token
    # TORGO examples contain sessions like wav_headMic_MC01S01 or wav_arrayMic_M03S02
    joined = "_".join(parts)
    # Capture both MC01/FC01 (controls) and M01/F03 style IDs.
    m = re.search(r"(?:^|_)([FM]C?\d{2}|C[FM]\d{2})(?:S\d{2})?(?:_|$)", joined)
    if m:
        return m.group(1)
    return "unknown"


def infer_severity_from_path(path: str, speaker_id: Optional[str] = None, dataset_name: Optional[str] = None) -> int:
    p = path.lower()
    # If folder names explicitly contain severity keywords, use them.
    if "very_low" in p:
        return SEVERITY_KEYWORDS["very_low"]
    if " low" in p or "_low" in p:
        return SEVERITY_KEYWORDS["low"]
    if "medium" in p or "moderate" in p or "_mid" in p:
        return SEVERITY_KEYWORDS["medium"]
    if "high" in p:
        return SEVERITY_KEYWORDS["high"]

    # UA Kaggle structure typically encodes severity at speaker level.
    if dataset_name == "UA":
        sid = speaker_id or infer_speaker_id_from_path(path)
        sid = sid.replace("C", "")  # controls become Fxx/Mxx after strip
        if sid in CONFIG["ua_speaker_severity_map"]:
            return int(CONFIG["ua_speaker_severity_map"][sid])
        return -1  # unknown/healthy/no severity label

    # TORGO usually does not include reliable severity labels in folder names.
    return -1


def infer_binary_label_from_path(path: str) -> int:
    """
    Robust binary labeling from path tokens.
    Avoid substring traps such as '/content' matching '/con'.
    """
    p = Path(path)
    parts_l = [x.lower() for x in p.parts]
    name_l = p.name.lower()

    # Strong folder-level indicators first.
    if any(tok in parts_l for tok in ["f_con", "m_con", "noisereduced-uaspeech-control"]):
        return 0
    if any(tok in parts_l for tok in ["f_dys", "m_dys", "noisereduced-uaspeech"]):
        return 1

    # Speaker folder indicators: CMxx/CFxx are controls; Mxx/Fxx are dys in UA dys folder.
    speaker = infer_speaker_id_from_path(path).upper()
    if re.fullmatch(r"(?:CM|CF)\d{2}", speaker):
        return 0
    if re.fullmatch(r"(?:M|F)\d{2}", speaker):
        return 1
    if re.fullmatch(r"(?:MC|FC)\d{2}", speaker):
        return 0

    # Generic fallback.
    full_l = "/".join(parts_l) + "/" + name_l
    if any(k in full_l for k in ["control", "healthy", "non_dys", "nondys"]):
        return 0
    if any(k in full_l for k in ["dys", "dysarth", "patient"]):
        return 1
    return 1


def collect_audio_files(root: str) -> List[str]:
    root_path = Path(root)
    if not root_path.exists():
        return []
    wav_files = list(root_path.rglob("*.wav"))
    return [str(p) for p in wav_files]


def diagnose_dataset_paths(config: Dict) -> None:
    print("\nDataset path diagnostics")
    print("------------------------")
    for dataset_name, root_key in [("TORGO", "torgo_root"), ("UA", "ua_root")]:
        root = Path(config[root_key])
        exists = root.exists()
        wav_count = len(list(root.rglob("*.wav"))) if exists else 0
        print(f"{dataset_name:>6} | root={root} | exists={exists} | wav_files={wav_count}")

        if exists and wav_count == 0:
            children = list(root.iterdir())[:8]
            preview = [p.name for p in children]
            print(f"      preview children: {preview if preview else '[]'}")


def _walk_dirs_bfs(root: Path, max_depth: int):
    """Breadth-first directory walk up to max_depth (0 = only root)."""
    queue: List[Tuple[Path, int]] = [(root.resolve(), 0)]
    while queue:
        d, depth = queue.pop(0)
        yield d, depth
        if depth >= max_depth:
            continue
        try:
            for child in sorted(d.iterdir()):
                if child.is_dir():
                    queue.append((child, depth + 1))
        except (PermissionError, OSError):
            pass


def _count_wavs_quick(root: Path, cap: int = 2000) -> int:
    n = 0
    try:
        for _ in root.rglob("*.wav"):
            n += 1
            if n >= cap:
                return n
    except OSError:
        return n
    return n


def _discover_best_folder(search_roots: List[str], keywords: Tuple[str, ...], max_depth: int) -> Optional[Path]:
    """Pick the directory under search_roots whose name matches a keyword and has the most .wav files."""
    best: Optional[Path] = None
    best_n = 0
    for sr in search_roots:
        base = Path(sr)
        if not base.exists():
            continue
        for d, _ in _walk_dirs_bfs(base, max_depth=max_depth):
            name_l = d.name.lower().replace("-", "_").replace(" ", "_")
            if not any(k in name_l for k in keywords):
                continue
            n = _count_wavs_quick(d, cap=1500)
            if n > best_n:
                best_n = n
                best = d
    return best if best_n > 0 else None


def autoconfigure_dataset_paths(config: Dict) -> None:
    """If TORGO/UA paths are missing or have no audio, try to find folders on Drive or /content."""
    if not config.get("auto_discover_dataset_paths", True):
        return
    roots = list(config.get("dataset_search_roots") or [])
    depth = int(config.get("dataset_search_max_depth", 6))

    def needs_fix(key: str) -> bool:
        p = Path(config[key])
        return (not p.exists()) or (_count_wavs_quick(p, cap=3) == 0)

    if needs_fix("torgo_root"):
        found = _discover_best_folder(roots, ("torgo",), depth)
        if found is not None:
            config["torgo_root"] = str(found)
            n_wav = _count_wavs_quick(found, cap=500)
            print(f"[auto-discover] TORGO -> {found} (quick scan: {n_wav} wav files, may be more)")
    if needs_fix("ua_root"):
        found = _discover_best_folder(
            roots,
            ("ua_speech", "uaspeech", "u_a_speech"),
            depth,
        )
        if found is not None:
            config["ua_root"] = str(found)
            n_wav = _count_wavs_quick(found, cap=500)
            print(f"[auto-discover] UA Speech -> {found} (quick scan: {n_wav} wav files, may be more)")


def setup_colab_drive_and_paths(config: Dict) -> None:
    """Optionally mount Google Drive (Colab only) then auto-fill dataset paths."""
    if not config.get("auto_mount_google_drive", False):
        print("Drive mount disabled in CONFIG. Using dataset paths as provided.")
        print("\nCurrent CONFIG dataset paths:")
        print(f"  torgo_root: {config['torgo_root']}")
        print(f"  ua_root:    {config['ua_root']}")
        return
    try:
        from google.colab import drive  # type: ignore

        if config.get("auto_mount_google_drive", False):
            print("Mounting Google Drive (authorize if prompted)...")
            drive.mount("/content/drive", force_remount=False)
    except ImportError:
        print("(Not in Google Colab: skipped Drive mount. Set CONFIG paths manually.)")
    autoconfigure_dataset_paths(config)
    print("\nCurrent CONFIG dataset paths:")
    print(f"  torgo_root: {config['torgo_root']}")
    print(f"  ua_root:    {config['ua_root']}")


# %% [markdown]
# ## 3b) Colab: mount Drive + auto-discover TORGO / UA Speech
#
# Run this cell after utilities are defined **or** move the call to just before `build_metadata`.
# If discovery finds nothing, upload/extract data under `/content/data/...` or set `CONFIG` paths manually.

# %%
setup_colab_drive_and_paths(CONFIG)

# %% [markdown]
# ## 4) Build metadata for TORGO + UA Speech
# 
# Priority:
# 1. Load `metadata_csv` if provided.
# 2. Else infer labels from directory/file naming conventions.
# 
# For robust research results, providing curated metadata CSV is strongly recommended.

# %%
def build_metadata(config: Dict) -> pd.DataFrame:
    if config["metadata_csv"] and Path(config["metadata_csv"]).exists():
        df = pd.read_csv(config["metadata_csv"])
        required_cols = {"audio_path", "dataset", "binary_label", "severity_label", "speaker_id"}
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"Metadata CSV missing required columns: {missing}")
        df = df.copy()
        return df

    rows = []
    for dataset_name, root_key in [("TORGO", "torgo_root"), ("UA", "ua_root")]:
        root = config[root_key]
        files = collect_audio_files(root)
        if config.get("max_samples_per_dataset") is not None and len(files) > int(config["max_samples_per_dataset"]):
            rng = np.random.RandomState(SEED)
            files = rng.choice(files, size=int(config["max_samples_per_dataset"]), replace=False).tolist()
            print(f"[build_metadata] {dataset_name}: sampled down to {len(files)} wav files for practicality")
        print(f"[build_metadata] {dataset_name}: found {len(files)} wav files at {root}")
        for fp in files:
            speaker_id = infer_speaker_id_from_path(fp)
            binary_label = infer_binary_label_from_path(fp)
            severity_label = infer_severity_from_path(fp, speaker_id=speaker_id, dataset_name=dataset_name)
            rows.append(
                {
                    "audio_path": fp,
                    "dataset": dataset_name,
                    "binary_label": int(binary_label),
                    "severity_label": int(severity_label),
                    "speaker_id": str(speaker_id),
                }
            )
    df = pd.DataFrame(rows)
    if df.empty:
        diagnose_dataset_paths(config)
        raise RuntimeError(
            "No audio files discovered.\n"
            "1) Put extracted TORGO and UA Speech (.wav) on Drive or under /content.\n"
            "2) Set CONFIG['torgo_root'] and CONFIG['ua_root'] to those folders (not a zip).\n"
            "3) Re-run the 'setup_colab_drive_and_paths' cell, or set CONFIG['dataset_search_roots'] "
            "to a parent folder that contains both datasets to speed up search.\n"
            "4) Optional: provide CONFIG['metadata_csv'] with an audio_path column."
        )
    return df


meta_df = build_metadata(CONFIG)
print(meta_df.head())
print("\nDataset counts:")
print(meta_df["dataset"].value_counts())
print("\nBinary label counts:")
print(meta_df["binary_label"].value_counts(dropna=False))
print("\nSeverity label counts (includes -1 = unknown/not-applicable):")
print(meta_df["severity_label"].value_counts(dropna=False).sort_index())

# %% [markdown]
# ## 5) Split strategy + imbalance handling
# 
# - Intra-dataset split (train/test by speaker to reduce leakage)
# - Cross-dataset split prepared separately
# - Imbalance handled with oversampling for baseline and class weights for wav2vec2

# %%
def speaker_stratified_split(df: pd.DataFrame, test_ratio: float = 0.2, seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.RandomState(seed)
    train_parts, test_parts = [], []
    for ds_name in df["dataset"].unique():
        ds_df = df[df["dataset"] == ds_name].copy()
        speakers = ds_df["speaker_id"].unique().tolist()
        rng.shuffle(speakers)
        n_test = max(1, int(len(speakers) * test_ratio))
        test_speakers = set(speakers[:n_test])
        ds_test = ds_df[ds_df["speaker_id"].isin(test_speakers)]
        ds_train = ds_df[~ds_df["speaker_id"].isin(test_speakers)]
        train_parts.append(ds_train)
        test_parts.append(ds_test)
    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    return train_df, test_df


def speaker_stratified_severity_split(
    df: pd.DataFrame, test_ratio: float = 0.2, seed: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split by speaker (no clip leakage) while keeping severity classes represented in train and test when possible.
    """
    if len(df) < 30 or df["speaker_id"].nunique() < 6:
        return speaker_stratified_split(df, test_ratio=test_ratio, seed=seed)
    n_splits = int(round(1.0 / max(0.05, min(0.5, test_ratio))))
    n_splits = max(2, min(n_splits, 10))
    try:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        X = np.zeros(len(df))
        y = df["severity_label"].to_numpy()
        groups = df["speaker_id"].to_numpy()
        train_idx, test_idx = next(sgkf.split(X, y, groups=groups))
        return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)
    except ValueError:
        return speaker_stratified_split(df, test_ratio=test_ratio, seed=seed)


def _coverage_counts(y: pd.Series, labels: List[int]) -> Dict[int, int]:
    vc = y.value_counts().to_dict()
    return {lb: int(vc.get(lb, 0)) for lb in labels}


def _has_full_coverage(train_df: pd.DataFrame, test_df: pd.DataFrame, labels: List[int]) -> bool:
    tr = _coverage_counts(train_df["severity_label"], labels)
    te = _coverage_counts(test_df["severity_label"], labels)
    return all(v > 0 for v in tr.values()) and all(v > 0 for v in te.values())


def collapse_severity_4_to_3(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse 4-level labels into 3 levels to improve split feasibility:
      0,1 -> 0 (low)
      2   -> 1 (medium)
      3   -> 2 (high)
    """
    out = df.copy()
    out["severity_label"] = out["severity_label"].map({0: 0, 1: 0, 2: 1, 3: 2}).astype(int)
    return out


def find_best_severity_split(
    df: pd.DataFrame,
    test_ratio: float,
    seed: int,
    expected_labels: List[int],
    max_tries: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, bool]:
    """
    Retry speaker-level severity split with different seeds until all expected classes
    are present in both train and test.
    """
    best_pair = None
    best_score = -1
    for k in range(max_tries):
        tr, te = speaker_stratified_severity_split(df, test_ratio=test_ratio, seed=seed + k)
        tr_cov = _coverage_counts(tr["severity_label"], expected_labels)
        te_cov = _coverage_counts(te["severity_label"], expected_labels)
        score = sum(int(v > 0) for v in tr_cov.values()) + sum(int(v > 0) for v in te_cov.values())
        if score > best_score:
            best_score = score
            best_pair = (tr, te)
        if _has_full_coverage(tr, te, expected_labels):
            return tr, te, True
    assert best_pair is not None
    return best_pair[0], best_pair[1], False


train_df, test_df = speaker_stratified_split(meta_df, test_ratio=0.2, seed=SEED)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print("Train binary distribution:\n", train_df["binary_label"].value_counts())

# Severity should use only rows with valid labels (>=0) and dysarthric class.
severity_df = meta_df[(meta_df["severity_label"] >= 0) & (meta_df["binary_label"] == 1)].reset_index(drop=True)
severity_note = ""
expected_labels = list(range(int(CONFIG.get("severity_expected_num_classes", 4))))
sev_train_df, sev_test_df, ok4 = find_best_severity_split(
    severity_df,
    test_ratio=0.2,
    seed=SEED,
    expected_labels=expected_labels,
    max_tries=int(CONFIG.get("severity_split_max_tries", 40)),
)
if not ok4 and CONFIG.get("severity_allow_collapse_4_to_3", True):
    print("[severity] Could not achieve full 4-class train/test coverage after retries. Falling back to 3-class severity.")
    severity_df = collapse_severity_4_to_3(severity_df)
    expected_labels = [0, 1, 2]
    sev_train_df, sev_test_df, ok3 = find_best_severity_split(
        severity_df,
        test_ratio=0.2,
        seed=SEED,
        expected_labels=expected_labels,
        max_tries=int(CONFIG.get("severity_split_max_tries", 40)),
    )
    if not ok3:
        severity_note = (
            "Severity split could not include all 3 classes in both train/test despite retries; "
            "severity metrics should be interpreted cautiously."
        )
    else:
        severity_note = "Severity labels were collapsed from 4 to 3 classes due to split coverage limits."
elif not ok4:
    severity_note = (
        "Severity split could not include all 4 classes in both train/test despite retries; "
        "severity metrics should be interpreted cautiously."
    )
if severity_note:
    print(f"[severity] {severity_note}")
print(f"Severity train/test sizes: {len(sev_train_df)} / {len(sev_test_df)}")
print("Severity train class distribution:\n", sev_train_df["severity_label"].value_counts().sort_index())
print("Severity test class distribution:\n", sev_test_df["severity_label"].value_counts().sort_index())

# %% [markdown]
# ## 6) Baseline: MFCC + SVM

# %%
def drop_unreadable_audio_rows(df: pd.DataFrame, desc: str = "Verify audio") -> pd.DataFrame:
    """Drop rows whose files cannot be decoded (keeps wav2vec training from crashing mid-epoch)."""
    keep: List[int] = []
    n_bad = 0
    for i, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            load_audio_mono_16k(row["audio_path"])
            keep.append(i)
        except Exception:
            n_bad += 1
    if n_bad:
        print(f"[drop_unreadable_audio_rows] Removed {n_bad} unreadable rows ({desc})")
    return df.loc[keep].reset_index(drop=True)


def extract_mfcc_features(path: str, n_mfcc: int = 40) -> np.ndarray:
    y = load_audio_mono_16k(path)
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    feat = np.concatenate([mfcc, delta, delta2], axis=0)
    stats = np.concatenate(
        [
            np.mean(feat, axis=1),
            np.std(feat, axis=1),
            np.min(feat, axis=1),
            np.max(feat, axis=1),
        ]
    )
    return stats.astype(np.float32)


def make_feature_matrix(df: pd.DataFrame, desc: str = "Extracting MFCC") -> Tuple[np.ndarray, pd.DataFrame]:
    """Return feature matrix and a row-aligned DataFrame (drops unreadable/corrupt audio)."""
    feats: List[np.ndarray] = []
    keep_rows: List[int] = []
    n_fail = 0
    first_fail: Optional[str] = None
    paths = df["audio_path"].tolist()
    for i, p in enumerate(tqdm(paths, desc=desc)):
        try:
            feats.append(extract_mfcc_features(p))
            keep_rows.append(i)
        except Exception as e:
            n_fail += 1
            if first_fail is None:
                first_fail = f"{p} ({e})"
    if not feats:
        raise RuntimeError(f"No MFCC features extracted. First failure: {first_fail}")
    if n_fail:
        print(f"[make_feature_matrix] Skipped {n_fail} files. Example: {first_fail}")
    aligned = df.iloc[keep_rows].reset_index(drop=True)
    return np.vstack(feats), aligned


X_train, train_df = make_feature_matrix(train_df, desc="MFCC train (binary)")
X_test, test_df = make_feature_matrix(test_df, desc="MFCC test (binary)")
y_train_bin = train_df["binary_label"].values
y_test_bin = test_df["binary_label"].values

X_train_sev, sev_train_df = make_feature_matrix(sev_train_df, desc="MFCC train (severity)")
X_test_sev, sev_test_df = make_feature_matrix(sev_test_df, desc="MFCC test (severity)")
y_train_sev = sev_train_df["severity_label"].values
y_test_sev = sev_test_df["severity_label"].values

# Baseline imbalance handling via random oversampling
ros = RandomOverSampler(random_state=SEED)
X_train_bin_bal, y_train_bin_bal = ros.fit_resample(X_train, y_train_bin)
X_train_sev_bal, y_train_sev_bal = ros.fit_resample(X_train_sev, y_train_sev)

bin_svm = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=5, gamma="scale", class_weight="balanced")),
    ]
)
bin_svm.fit(X_train_bin_bal, y_train_bin_bal)
bin_pred = bin_svm.predict(X_test)
bin_metrics_baseline = metric_bundle(y_test_bin, bin_pred, average="binary")
print_metrics_table("Baseline MFCC+SVM (Binary)", bin_metrics_baseline)

print("\nBinary classification report:")
print(classification_report(y_test_bin, bin_pred, digits=4))

sev_svm = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=5, gamma="scale", class_weight="balanced")),
    ]
)
sev_svm.fit(X_train_sev_bal, y_train_sev_bal)
sev_pred = sev_svm.predict(X_test_sev)
sev_metrics_baseline = metric_bundle(y_test_sev, sev_pred, average="macro")
print_metrics_table("Baseline MFCC+SVM (Severity)", sev_metrics_baseline)

print("\nSeverity classification report:")
print(classification_report(y_test_sev, sev_pred, digits=4))

# %% [markdown]
# ## 7) wav2vec2 dataset and loaders

# %%
class DysarthriaAudioDataset(Dataset):
    def __init__(self, df: pd.DataFrame, label_col: str):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        x = load_audio_mono_16k(row["audio_path"])
        y = int(row[self.label_col])
        return {"input_values": x, "labels": y}


@dataclass
class Collator:
    feature_extractor: AutoFeatureExtractor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        inputs = [f["input_values"] for f in features]
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        batch = self.feature_extractor(
            inputs,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )
        batch["labels"] = labels
        return batch


def make_loader(df: pd.DataFrame, label_col: str, feature_extractor, shuffle: bool = True) -> DataLoader:
    ds = DysarthriaAudioDataset(df, label_col=label_col)
    nw = int(CONFIG["num_workers"])
    kw: Dict = dict(
        dataset=ds,
        batch_size=CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=nw,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=Collator(feature_extractor),
    )
    if nw > 0:
        kw["prefetch_factor"] = 2
    return DataLoader(**kw)

# %% [markdown]
# ## 8) wav2vec2 model setup and training functions

# %%
def build_wav2vec2_model(num_labels: int) -> Wav2Vec2ForSequenceClassification:
    load_kw: Dict[str, Any] = dict(
        num_labels=num_labels,
        problem_type="single_label_classification",
        attention_dropout=0.1,
        hidden_dropout=0.1,
        feat_proj_dropout=0.1,
    )
    impl = CONFIG.get("wav2vec_attn_implementation") or "eager"
    if impl and str(impl).lower() not in ("auto", "none", ""):
        load_kw["attn_implementation"] = impl
    try:
        model = Wav2Vec2ForSequenceClassification.from_pretrained(
            CONFIG["wav2vec_checkpoint"],
            **load_kw,
        )
    except TypeError:
        load_kw.pop("attn_implementation", None)
        model = Wav2Vec2ForSequenceClassification.from_pretrained(
            CONFIG["wav2vec_checkpoint"],
            **load_kw,
        )
    if CONFIG["freeze_feature_encoder"]:
        model.wav2vec2.feature_extractor._freeze_parameters()
    n_freeze = CONFIG["freeze_encoder_layers"]
    for i, layer in enumerate(model.wav2vec2.encoder.layers):
        if i < n_freeze:
            for p in layer.parameters():
                p.requires_grad = False
    return model


def class_weights_from_labels(labels: np.ndarray) -> torch.Tensor:
    classes, counts = np.unique(labels, return_counts=True)
    total = counts.sum()
    weights = np.zeros(classes.max() + 1, dtype=np.float32)
    for c, cnt in zip(classes, counts):
        weights[c] = total / (len(classes) * cnt)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def train_one_task(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    label_col: str,
    num_labels: int,
    epochs: int,
    average_mode: str,
    run_name: str,
) -> Tuple[Wav2Vec2ForSequenceClassification, Dict]:
    feature_extractor = AutoFeatureExtractor.from_pretrained(CONFIG["wav2vec_checkpoint"])
    model = build_wav2vec2_model(num_labels).to(DEVICE)

    train_loader = make_loader(train_df, label_col, feature_extractor, shuffle=True)
    valid_loader = make_loader(valid_df, label_col, feature_extractor, shuffle=False)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    accum = max(1, int(CONFIG.get("gradient_accumulation_steps", 1)))
    steps_per_epoch = math.ceil(len(train_loader) / accum)
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    cweights = class_weights_from_labels(train_df[label_col].values)
    criterion = nn.CrossEntropyLoss(weight=cweights)

    use_amp = bool(CONFIG.get("use_amp", True)) and DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if use_amp else None

    best_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(
            tqdm(train_loader, desc=f"[{run_name}] Train epoch {epoch}/{epochs}")
        ):
            input_values = batch["input_values"].to(DEVICE)
            attention_mask = batch.get("attention_mask")
            if attention_mask is not None:
                attention_mask = attention_mask.to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    out = model(input_values=input_values, attention_mask=attention_mask)
                    loss = criterion(out.logits, labels) / accum
                scaler.scale(loss).backward()
            else:
                out = model(input_values=input_values, attention_mask=attention_mask)
                loss = criterion(out.logits, labels) / accum
                loss.backward()
            train_loss += loss.detach().float().item() * accum

            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        model.eval()
        val_loss = 0.0
        y_true, y_pred = [], []
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc=f"[{run_name}] Valid epoch {epoch}/{epochs}"):
                input_values = batch["input_values"].to(DEVICE)
                attention_mask = batch.get("attention_mask")
                if attention_mask is not None:
                    attention_mask = attention_mask.to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                if use_amp:
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        out = model(input_values=input_values, attention_mask=attention_mask)
                        loss = criterion(out.logits, labels)
                else:
                    out = model(input_values=input_values, attention_mask=attention_mask)
                    loss = criterion(out.logits, labels)
                val_loss += loss.item()

                preds = torch.argmax(out.logits.float(), dim=-1)
                y_true.extend(labels.cpu().numpy().tolist())
                y_pred.extend(preds.cpu().numpy().tolist())

        m = metric_bundle(y_true, y_pred, average=average_mode)
        epoch_log = {
            "epoch": epoch,
            "train_loss": train_loss / max(1, len(train_loader)),
            "val_loss": val_loss / max(1, len(valid_loader)),
            **m,
        }
        history.append(epoch_log)
        print(
            f"[{run_name}] Epoch {epoch} | "
            f"train_loss={epoch_log['train_loss']:.4f} "
            f"val_loss={epoch_log['val_loss']:.4f} "
            f"acc={epoch_log['accuracy']:.4f} f1={epoch_log['f1']:.4f}"
        )

        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {"history": history, "best_f1": best_f1}

# %% [markdown]
# ## 9) Train wav2vec2 - binary and severity

# %%
bin_model, bin_train_log = train_one_task(
    train_df=train_df,
    valid_df=test_df,
    label_col="binary_label",
    num_labels=2,
    epochs=CONFIG["epochs_binary"],
    average_mode="binary",
    run_name="wav2vec2_binary",
)

if len(severity_df) == 0:
    raise RuntimeError(
        "No valid severity samples found. Check CONFIG['ua_speaker_severity_map'] "
        "or provide metadata CSV with severity labels."
    )
sev_num_labels = int(severity_df["severity_label"].max() + 1)
sev_model, sev_train_log = train_one_task(
    train_df=sev_train_df,
    valid_df=sev_test_df,
    label_col="severity_label",
    num_labels=sev_num_labels,
    epochs=CONFIG["epochs_severity"],
    average_mode="macro",
    run_name="wav2vec2_severity",
)

# %% [markdown]
# ## 10) Evaluate wav2vec2 on held-out test

# %%
def evaluate_wav2vec2(
    model: Wav2Vec2ForSequenceClassification,
    df: pd.DataFrame,
    label_col: str,
    average_mode: str,
) -> Dict:
    feature_extractor = AutoFeatureExtractor.from_pretrained(CONFIG["wav2vec_checkpoint"])
    loader = make_loader(df, label_col, feature_extractor, shuffle=False)
    model.eval()
    y_true, y_pred = [], []
    use_amp = bool(CONFIG.get("use_amp", True)) and DEVICE == "cuda"
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Eval {label_col}"):
            x = batch["input_values"].to(DEVICE)
            m = batch.get("attention_mask")
            if m is not None:
                m = m.to(DEVICE)
            y = batch["labels"].to(DEVICE)
            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    out = model(input_values=x, attention_mask=m)
            else:
                out = model(input_values=x, attention_mask=m)
            pred = torch.argmax(out.logits.float(), dim=-1)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
    metrics = metric_bundle(y_true, y_pred, average=average_mode)
    return {"metrics": metrics, "y_true": y_true, "y_pred": y_pred}


bin_eval = evaluate_wav2vec2(bin_model, test_df, "binary_label", "binary")
sev_eval = evaluate_wav2vec2(sev_model, sev_test_df, "severity_label", "macro")

print_metrics_table("wav2vec2 (Binary)", bin_eval["metrics"])
print("\nBinary report:\n", classification_report(bin_eval["y_true"], bin_eval["y_pred"], digits=4))

print_metrics_table("wav2vec2 (Severity)", sev_eval["metrics"])
print("\nSeverity report:\n", classification_report(sev_eval["y_true"], sev_eval["y_pred"], digits=4))

# %% [markdown]
# ## 11) Cross-dataset evaluation
# 
# - Train on TORGO, test on UA
# - Train on UA, test on TORGO

# %%
def cross_dataset_split(df: pd.DataFrame, train_dataset: str, test_dataset: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    tr = df[df["dataset"] == train_dataset].reset_index(drop=True)
    te = df[df["dataset"] == test_dataset].reset_index(drop=True)
    if len(tr) == 0 or len(te) == 0:
        raise ValueError(f"Missing data for train={train_dataset} or test={test_dataset}")
    return tr, te


def run_cross_eval_baseline(df: pd.DataFrame, train_ds: str, test_ds: str, label_col: str, avg: str) -> Dict:
    tr, te = cross_dataset_split(df, train_ds, test_ds)
    X_tr, tr_ok = make_feature_matrix(tr, desc=f"MFCC cross-train {train_ds}->{test_ds}")
    X_te, te_ok = make_feature_matrix(te, desc=f"MFCC cross-test {train_ds}->{test_ds}")
    y_tr = tr_ok[label_col].values
    y_te = te_ok[label_col].values

    ros = RandomOverSampler(random_state=SEED)
    X_tr_b, y_tr_b = ros.fit_resample(X_tr, y_tr)
    clf = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=5, gamma="scale", class_weight="balanced")),
        ]
    )
    clf.fit(X_tr_b, y_tr_b)
    y_pred = clf.predict(X_te)
    return {"metrics": metric_bundle(y_te, y_pred, average=avg), "y_true": y_te, "y_pred": y_pred}


def run_cross_eval_w2v(
    df: pd.DataFrame,
    train_ds: str,
    test_ds: str,
    label_col: str,
    num_labels: int,
    avg: str,
    epochs: int,
    run_name: str,
) -> Dict:
    tr, te = cross_dataset_split(df, train_ds, test_ds)
    tr = drop_unreadable_audio_rows(tr, desc=f"cross w2v train {train_ds}")
    te = drop_unreadable_audio_rows(te, desc=f"cross w2v test {test_ds}")
    if len(tr) == 0 or len(te) == 0:
        return {"metrics": {"accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan}}
    model, _ = train_one_task(
        train_df=tr,
        valid_df=te,
        label_col=label_col,
        num_labels=num_labels,
        epochs=epochs,
        average_mode=avg,
        run_name=run_name,
    )
    return evaluate_wav2vec2(model, te, label_col, avg)


cross_results = {}
for train_ds, test_ds in [("TORGO", "UA"), ("UA", "TORGO")]:
    key = f"{train_ds}_to_{test_ds}"
    cross_results[key] = {}
    cross_results[key]["baseline_binary"] = run_cross_eval_baseline(
        meta_df, train_ds, test_ds, label_col="binary_label", avg="binary"
    )
    sev_meta = meta_df[(meta_df["severity_label"] >= 0) & (meta_df["binary_label"] == 1)].reset_index(drop=True)
    tr_sev = sev_meta[sev_meta["dataset"] == train_ds]
    te_sev = sev_meta[sev_meta["dataset"] == test_ds]
    if len(tr_sev) > 0 and len(te_sev) > 0:
        cross_results[key]["baseline_severity"] = run_cross_eval_baseline(
            sev_meta, train_ds, test_ds, label_col="severity_label", avg="macro"
        )
    else:
        cross_results[key]["baseline_severity"] = {"metrics": {"accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan}}
    cross_results[key]["w2v_binary"] = run_cross_eval_w2v(
        meta_df, train_ds, test_ds, "binary_label", 2, "binary", max(2, CONFIG["epochs_binary"] - 1), f"w2v_bin_{key}"
    )
    if len(tr_sev) > 0 and len(te_sev) > 0:
        cross_results[key]["w2v_severity"] = run_cross_eval_w2v(
            sev_meta, train_ds, test_ds, "severity_label", sev_num_labels, "macro", max(2, CONFIG["epochs_severity"] - 1), f"w2v_sev_{key}"
        )
    else:
        cross_results[key]["w2v_severity"] = {"metrics": {"accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan}}

for k, v in cross_results.items():
    print(f"\n=== Cross Eval: {k} ===")
    for model_name, pack in v.items():
        print_metrics_table(model_name, pack["metrics"])

# %% [markdown]
# ## 12) Interpretability
# 
# We include:
# 1. **SVM feature importance proxy** via permutation importance (optional; can be slow)
# 2. **wav2vec2 saliency** — gradient magnitude on the input waveform
# 3. **wav2vec2 attention rollout** — aggregated self-attention over encoder layers, upsampled to approximate time

# %%
from sklearn.inspection import permutation_importance

if CONFIG.get("enable_interpretability", False):
    x_pi = X_test
    y_pi = y_test_bin
    max_pi = int(CONFIG.get("perm_max_samples", 0) or 0)
    if max_pi > 0 and len(X_test) > max_pi:
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(X_test), size=max_pi, replace=False)
        x_pi = X_test[idx]
        y_pi = y_test_bin[idx]
        print(f"[interpretability] Using subset for permutation importance: {len(x_pi)} samples")

    perm = permutation_importance(
        estimator=bin_svm,
        X=x_pi,
        y=y_pi,
        n_repeats=int(CONFIG.get("perm_n_repeats", 1)),
        random_state=SEED,
        scoring="f1",
        n_jobs=int(CONFIG.get("perm_n_jobs", -1)),
    )
    importances = perm.importances_mean

    plt.figure(figsize=(10, 4))
    plt.plot(importances)
    plt.title("MFCC+SVM Permutation Importance (Binary)")
    plt.xlabel("Feature Index")
    plt.ylabel("Importance (Mean drop in F1)")
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("[interpretability] Skipped (set CONFIG['enable_interpretability']=True to run).")

# %%
def wav2vec_attention_rollout(
    model: Wav2Vec2ForSequenceClassification,
    input_values: torch.Tensor,
    attention_mask: Optional[torch.Tensor],
) -> Tuple[torch.Tensor, Tuple[torch.Tensor, ...]]:
    """
    Attention rollout over encoder layers (mean over heads per layer, residual + renormalize).
    Returns per-encoder-frame importance (length T) and raw attention tuple for optional heatmaps.
    Reference: Abnar & Zuidema (2020); Chefer et al. style identity residual.
    """
    setter = getattr(model, "set_attn_implementation", None)
    if callable(setter):
        setter("eager")

    with torch.no_grad():
        out = model(
            input_values,
            attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True,
        )
    attentions = out.attentions
    if attentions is None or len(attentions) == 0:
        raise RuntimeError("No attentions returned; use a Wav2Vec2 model with output_attentions=True.")

    layer_attns = [a for a in attentions if a is not None]
    if not layer_attns:
        raise RuntimeError(
            "Attention weights are all None (common with SDPA/FlashAttention). "
            "Set CONFIG['wav2vec_attn_implementation']='eager', rebuild the model, and re-run training — "
            "or re-run the cell after updating so set_attn_implementation('eager') applies."
        )

    t_len = layer_attns[0].shape[-1]
    eye = torch.eye(t_len, device=input_values.device, dtype=layer_attns[0].dtype)
    rollout_mat = eye.clone()
    for attn in layer_attns:
        a = attn[0].mean(dim=0)
        a = a + eye
        a = a / a.sum(dim=-1, keepdim=True).clamp(min=1e-9)
        rollout_mat = torch.matmul(a, rollout_mat)
    importance = rollout_mat.mean(dim=0)
    importance = importance / (importance.sum() + 1e-9)
    return importance, tuple(layer_attns)


def upsample_encoder_importance_to_audio(
    importance: np.ndarray,
    num_audio_samples: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Map encoder-frame importance to one value per audio sample (linear interp)."""
    t_enc = importance.shape[0]
    x_old = np.linspace(0, num_audio_samples - 1, num=t_enc)
    x_new = np.arange(num_audio_samples, dtype=np.float64)
    up = np.interp(x_new, x_old, importance.astype(np.float64))
    up = np.maximum(up, 0.0)
    up = up / (np.sum(up) + 1e-9)
    times = x_new / float(TARGET_SR)
    return times, up


def plot_wav2vec_attention_maps(
    model: Wav2Vec2ForSequenceClassification,
    audio_path: str,
    save_prefix: Optional[Path] = None,
) -> None:
    """Waveform, attention-rollout importance over time, and last-layer attention heatmap."""
    model.eval()
    feat = AutoFeatureExtractor.from_pretrained(CONFIG["wav2vec_checkpoint"])
    x = load_audio_mono_16k(audio_path)
    batch = feat([x], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    input_values = batch["input_values"].to(DEVICE)
    attention_mask = batch.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    imp_t, attn_tuple = wav2vec_attention_rollout(model, input_values, attention_mask)

    imp_np = imp_t.detach().float().cpu().numpy()
    times_a, imp_audio = upsample_encoder_importance_to_audio(imp_np, len(x))

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), constrained_layout=True)
    t_audio = np.arange(len(x)) / float(TARGET_SR)
    axes[0].plot(t_audio, x, color="gray", alpha=0.7, lw=0.5)
    axes[0].set_title(f"Waveform — {Path(audio_path).name}")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_xlabel("Time (s)")

    axes[1].fill_between(times_a, 0, imp_audio, alpha=0.6, color="C0")
    axes[1].plot(times_a, imp_audio, color="C0", lw=1)
    axes[1].set_title("Attention rollout importance (upsampled to audio time)")
    axes[1].set_ylabel("Normalized mass")
    axes[1].set_xlabel("Time (s)")

    last = attn_tuple[-1][0].mean(dim=0).float().cpu().numpy()
    max_show = 256
    if last.shape[0] > max_show:
        step = last.shape[0] // max_show
        last = last[::step, ::step]
    im = axes[2].imshow(last, aspect="auto", origin="lower", cmap="magma")
    axes[2].set_title("Last layer self-attention (mean over heads; subsampled if large)")
    axes[2].set_xlabel("Key frame")
    axes[2].set_ylabel("Query frame")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    if save_prefix is not None:
        out_path = Path(save_prefix).with_suffix(".png")
        fig.savefig(out_path, dpi=150)
        print(f"[attention] Saved figure: {out_path}")
    plt.show()


def wav2vec_saliency(
    model: Wav2Vec2ForSequenceClassification,
    audio_path: str,
    target_label: Optional[int] = None,
    window_ms: int = 50,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    feat = AutoFeatureExtractor.from_pretrained(CONFIG["wav2vec_checkpoint"])
    x = load_audio_mono_16k(audio_path)
    batch = feat([x], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    input_values = batch["input_values"].to(DEVICE)
    input_values.requires_grad_(True)
    attention_mask = batch.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    out = model(input_values=input_values, attention_mask=attention_mask)
    logits = out.logits
    if target_label is None:
        target_label = int(torch.argmax(logits, dim=-1).item())

    score = logits[0, target_label]
    model.zero_grad()
    score.backward()
    saliency = input_values.grad.detach().abs().squeeze().cpu().numpy()

    step = int(TARGET_SR * window_ms / 1000)
    step = max(step, 1)
    pooled = np.array([saliency[i : i + step].mean() for i in range(0, len(saliency), step)])
    times = np.arange(len(pooled)) * (window_ms / 1000.0)
    return times, pooled


if CONFIG.get("enable_interpretability", False):
    sample_path = test_df.iloc[0]["audio_path"]
    t, s = wav2vec_saliency(bin_model, sample_path)
    plt.figure(figsize=(12, 3))
    plt.plot(t, s)
    plt.title(f"wav2vec2 Saliency over Time\nSample: {Path(sample_path).name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Saliency")
    plt.grid(True, alpha=0.3)
    plt.show()

# Attention rollout maps (optional, fast)
if CONFIG.get("enable_attention_maps", True):
    n_att = max(1, int(CONFIG.get("attention_num_samples", 1)))
    save_att = bool(CONFIG.get("attention_save_plots", True))
    out_dir = Path(CONFIG["output_dir"])
    for i in range(min(n_att, len(test_df))):
        p = test_df.iloc[i]["audio_path"]
        prefix = out_dir / f"attention_map_bin_sample{i}" if save_att else None
        plot_wav2vec_attention_maps(bin_model, p, save_prefix=prefix)
else:
    print("[attention] Skipped (set CONFIG['enable_attention_maps']=True).")

# %% [markdown]
# ## 13) Compare baseline vs wav2vec2

# %%
comparison_rows = [
    {
        "setting": "intra_dataset_test",
        "task": "binary",
        "model": "MFCC+SVM",
        **bin_metrics_baseline,
    },
    {
        "setting": "intra_dataset_test",
        "task": "binary",
        "model": "wav2vec2",
        **bin_eval["metrics"],
    },
    {
        "setting": "intra_dataset_test",
        "task": "severity",
        "model": "MFCC+SVM",
        **sev_metrics_baseline,
    },
    {
        "setting": "intra_dataset_test",
        "task": "severity",
        "model": "wav2vec2",
        **sev_eval["metrics"],
    },
]

for split_name, split_data in cross_results.items():
    comparison_rows.append(
        {
            "setting": split_name,
            "task": "binary",
            "model": "MFCC+SVM",
            **split_data["baseline_binary"]["metrics"],
        }
    )
    comparison_rows.append(
        {
            "setting": split_name,
            "task": "binary",
            "model": "wav2vec2",
            **split_data["w2v_binary"]["metrics"],
        }
    )
    comparison_rows.append(
        {
            "setting": split_name,
            "task": "severity",
            "model": "MFCC+SVM",
            **split_data["baseline_severity"]["metrics"],
        }
    )
    comparison_rows.append(
        {
            "setting": split_name,
            "task": "severity",
            "model": "wav2vec2",
            **split_data["w2v_severity"]["metrics"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

plot_df = comparison_df[pd.notna(comparison_df["f1"])].copy()
if plot_df.empty:
    print("[comparison plot] Skipped: all F1 values are NaN.")
else:
    plt.figure(figsize=(12, 5))
    sns.barplot(data=plot_df, x="setting", y="f1", hue="model")
    plt.title("F1 Comparison Across Settings")
    plt.xticks(rotation=20)
    plt.grid(axis="y", alpha=0.3)
    plt.show()

comparison_csv = Path(CONFIG["output_dir"]) / "model_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)
print(f"Saved comparison table to: {comparison_csv}")

# %% [markdown]
# ## 14) Auto-generate structured report

# %%
def build_report_text(comparison: pd.DataFrame, severity_note_text: str = "") -> str:
    def best_line(task: str, setting: str) -> str:
        sub = comparison[(comparison["task"] == task) & (comparison["setting"] == setting)]
        if sub.empty:
            return f"- No result available for {task} / {setting}."
        valid = sub[pd.notna(sub["f1"])]
        if valid.empty:
            return f"- {setting} / {task}: metrics unavailable (F1 is NaN for all models, e.g. missing cross-dataset severity)."
        best_idx = valid["f1"].idxmax()
        if pd.isna(best_idx):
            return f"- {setting} / {task}: could not pick best model (invalid F1)."
        row = valid.loc[best_idx]
        acc = row["accuracy"]
        acc_s = f"{acc:.4f}" if pd.notna(acc) else "nan"
        return (
            f"- {setting} / {task}: best model={row['model']}, "
            f"F1={row['f1']:.4f}, Accuracy={acc_s}"
        )

    text = f"""# Dysarthria Detection and Severity Classification Report

## Objective
Build an end-to-end pipeline for automatic dysarthria detection (binary) and severity classification (multi-class) using TORGO and UA Speech datasets, and compare a classical baseline vs a wav2vec2 deep model.

## Methodology
- Data preparation: loading TORGO and UA Speech, resampling to 16kHz, fixed-length padding/trimming ({MAX_AUDIO_SEC:.1f}s), normalization.
- Baseline model: MFCC + delta + delta-delta statistical features, SVM classifier.
- Deep model: pretrained wav2vec2 fine-tuning with partial layer freezing, AdamW optimizer, linear warmup scheduler.
- Evaluation: intra-dataset split (speaker-level) and cross-dataset transfer (TORGO->UA and UA->TORGO).
- Interpretability: MFCC permutation importance and wav2vec2 gradient saliency over time.

## Results
{best_line("binary", "intra_dataset_test")}
{best_line("severity", "intra_dataset_test")}
{best_line("binary", "TORGO_to_UA")}
{best_line("severity", "TORGO_to_UA")}
{best_line("binary", "UA_to_TORGO")}
{best_line("severity", "UA_to_TORGO")}

## Comparison
- wav2vec2 typically improves representation quality and often outperforms MFCC+SVM in F1, especially for binary dysarthria detection.
- MFCC+SVM remains lightweight and can be competitive on smaller splits.
- Cross-dataset performance is usually lower than intra-dataset due to domain mismatch (speaker profiles, recording conditions, prompts).

## Limitations
- Label inference from folder names may be noisy unless a curated metadata CSV is used.
- Limited epochs and fixed audio duration are chosen for Colab practicality and may underfit.
- Severity classes can be imbalanced and inconsistently defined across datasets.
- {severity_note_text if severity_note_text else "Severity labels may be approximate in Kaggle-packaged datasets (speaker-level mapping)."}

## Conclusion
This pipeline provides a practical, reproducible baseline-to-deep-learning workflow for dysarthria analysis. For production or publication-grade performance, improve metadata quality, run stronger hyperparameter search, and add domain adaptation techniques.
"""
    return text


report_text = build_report_text(comparison_df, severity_note)
report_path = Path(CONFIG["output_dir"]) / "final_report.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)
print(f"\nReport saved to: {report_path}")

# %% [markdown]
# ## 15) Save trained models and logs

# %%
binary_model_path = Path(CONFIG["output_dir"]) / "wav2vec2_binary_best.pt"
severity_model_path = Path(CONFIG["output_dir"]) / "wav2vec2_severity_best.pt"
torch.save(bin_model.state_dict(), binary_model_path)
torch.save(sev_model.state_dict(), severity_model_path)

logs_path = Path(CONFIG["output_dir"]) / "training_logs.json"
payload = {
    "binary_train_log": bin_train_log,
    "severity_train_log": sev_train_log,
    "cross_results_metrics_only": {
        k: {mk: mv["metrics"] for mk, mv in v.items()} for k, v in cross_results.items()
    },
}
with open(logs_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print(f"Saved binary model:   {binary_model_path}")
print(f"Saved severity model: {severity_model_path}")
print(f"Saved logs:           {logs_path}")

